# CSV Basics — 03: Schema Inference vs Explicit Schema

## What you will learn
Schema inference is convenient but dangerous in production.
This notebook explains exactly what happens under the hood and
teaches you to always define schemas explicitly.

In this notebook:
1. How `inferSchema` works internally (two-pass scan)
2. Performance cost of inferSchema on large files
3. `samplingRatio` — the dangerous middle ground
4. Building explicit schemas: StructType, DDL string, JSON schema
5. Schema validation — catching mismatches early
6. Schema evolution — handling multiple CSV versions


In [1]:
import os, time, pathlib, shutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/csv_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('csv-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | DATA_DIR: {DATA_DIR}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 19:24:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2 | DATA_DIR: /workspace/data/csv_basics


In [2]:
import pathlib, time, random, datetime

random.seed(42)
N = 100_000

# Generate a realistic CSV with mixed types
rows = ["id,name,email,age,salary,hire_date,is_manager,score"]
for i in range(N):
    hdate = datetime.date(2015, 1, 1) + datetime.timedelta(days=random.randint(0, 3000))
    rows.append(
        f"{i+1},Employee_{i+1},emp{i+1}@company.com,"
        f"{random.randint(22,65)},"
        f"{round(random.gauss(75000,20000),2)},"
        f"{hdate},"
        f"{str(random.random()>0.8).lower()},"
        f"{round(random.uniform(1,10),2)}"
    )

csv_path = f"{DATA_DIR}/employees_large.csv"
pathlib.Path(csv_path).write_text("\n".join(rows))
print(f"Generated: {N:,} rows → {csv_path}")
size_mb = pathlib.Path(csv_path).stat().st_size / 1024 / 1024
print(f"File size: {size_mb:.1f} MB")

Generated: 100,000 rows → /workspace/data/csv_basics/employees_large.csv
File size: 7.2 MB


## Step 1 — The Cost of inferSchema

`inferSchema=True` reads the file **twice**:
1. First pass: scan all values to determine types
2. Second pass: actually read and convert values

This means every `inferSchema` read takes **2x the I/O**.


In [3]:
csv_path = f"{DATA_DIR}/employees_large.csv"

# Without inferSchema — fast but all strings
t0 = time.time()
df_no_infer = spark.read.csv(csv_path, header=True)
df_no_infer.count()   # trigger execution
t_no_infer = time.time() - t0
print(f"Without inferSchema : {t_no_infer:.3f}s | all StringType")

# With inferSchema — slower (2x file scan)
t0 = time.time()
df_inferred = spark.read.csv(csv_path, header=True, inferSchema=True)
df_inferred.count()
t_infer = time.time() - t0
print(f"With inferSchema    : {t_infer:.3f}s | types guessed")
print(f"Overhead            : {t_infer/t_no_infer:.1f}x slower")
print()
print("Inferred schema:")
df_inferred.printSchema()

Without inferSchema : 4.694s | all StringType


With inferSchema    : 1.795s | types guessed
Overhead            : 0.4x slower

Inferred schema:
root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- salary: double (nullable = true)
 |-- hire_date: date (nullable = true)
 |-- is_manager: boolean (nullable = true)
 |-- score: double (nullable = true)



## Step 2 — samplingRatio: Dangerous Shortcut

`samplingRatio=0.1` reads only 10% of the file to guess types.
Faster but dangerous — if the sample misses certain values, types are wrong.


In [4]:
# Create a CSV where first 10% has only integers, then floats appear
tricky_rows = ["id,value"]
for i in range(900):
    tricky_rows.append(f"{i},100")          # integers — in first 90%
for i in range(100):
    tricky_rows.append(f"{i+900},3.14")     # floats — only in last 10%

tricky_path = f"{DATA_DIR}/tricky_types.csv"
pathlib.Path(tricky_path).write_text("\n".join(tricky_rows))

# Sample only 10% — misses the float values!
df_sampled = spark.read.csv(tricky_path, header=True,
                              inferSchema=True, samplingRatio=0.1)
print("samplingRatio=0.1 — only sees integer values in sample:")
df_sampled.printSchema()

# What happens when we try to read the floats?
try:
    df_sampled.agg({"value": "sum"}).collect()
    print("Sum computed (integer overflow or wrong result!)")
except Exception as e:
    print(f"Error: {e}")

print()
# Full scan — catches floats
df_full = spark.read.csv(tricky_path, header=True, inferSchema=True)
print("Full inferSchema — detects float:")
df_full.printSchema()

samplingRatio=0.1 — only sees integer values in sample:
root
 |-- id: integer (nullable = true)
 |-- value: double (nullable = true)

Sum computed (integer overflow or wrong result!)

Full inferSchema — detects float:
root
 |-- id: integer (nullable = true)
 |-- value: double (nullable = true)



## Step 3 — Building Explicit Schemas

Three ways to define a schema:
1. **StructType + StructField** — most explicit, IDE-friendly
2. **DDL string** — concise, readable
3. **JSON schema** — portable, load from file


In [5]:
from pyspark.sql.types import *

# Method 1: StructType (most explicit)
schema_structtype = StructType([
    StructField("id",          IntegerType(),  nullable=False),
    StructField("name",        StringType(),   nullable=False),
    StructField("email",       StringType(),   nullable=True),
    StructField("age",         IntegerType(),  nullable=True),
    StructField("salary",      DoubleType(),   nullable=True),
    StructField("hire_date",   DateType(),     nullable=True),
    StructField("is_manager",  BooleanType(),  nullable=True),
    StructField("score",       FloatType(),    nullable=True),
])
print("Method 1: StructType")
print(schema_structtype.simpleString())

# Method 2: DDL string (concise)
schema_ddl = """
    id         INT     NOT NULL,
    name       STRING  NOT NULL,
    email      STRING,
    age        INT,
    salary     DOUBLE,
    hire_date  DATE,
    is_manager BOOLEAN,
    score      FLOAT
"""
schema_from_ddl = spark.sql(f"SELECT * FROM VALUES (1) tmp").schema  # just to verify
schema_from_ddl = StructType.fromDDL(schema_ddl.strip())
print("\nMethod 2: DDL string")
print(schema_from_ddl.simpleString())

# Method 3: JSON schema (portable — store in file)
import json as pyjson
schema_json = schema_structtype.json()
pathlib.Path(f"{DATA_DIR}/schema.json").write_text(schema_json)
schema_from_json = StructType.fromJson(pyjson.loads(schema_json))
print("\nMethod 3: JSON schema (saved to schema.json)")
print(f"  Columns: {[f.name for f in schema_from_json.fields]}")

Method 1: StructType
struct<id:int,name:string,email:string,age:int,salary:double,hire_date:date,is_manager:boolean,score:float>

Method 2: DDL string
struct<id:int,name:string,email:string,age:int,salary:double,hire_date:date,is_manager:boolean,score:float>

Method 3: JSON schema (saved to schema.json)
  Columns: ['id', 'name', 'email', 'age', 'salary', 'hire_date', 'is_manager', 'score']


## Step 4 — Applying Explicit Schema and Handling Mismatches

When schema doesn't match data, Spark produces nulls or errors depending on mode.


In [6]:
csv_path = f"{DATA_DIR}/employees_large.csv"

# Apply explicit schema — fast single pass
t0 = time.time()
df_explicit = spark.read.csv(csv_path, header=True,
                              schema=schema_structtype,
                              dateFormat="yyyy-MM-dd")
df_explicit.count()
t_explicit = time.time() - t0

print(f"Explicit schema read: {t_explicit:.3f}s")
print()
df_explicit.printSchema()
df_explicit.show(5)
print(f"\nis_manager true count: {df_explicit.filter(F.col('is_manager')==True).count()}")

Explicit schema read: 0.262s

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- salary: double (nullable = true)
 |-- hire_date: date (nullable = true)
 |-- is_manager: boolean (nullable = true)
 |-- score: float (nullable = true)

+---+----------+----------------+---+--------+----------+----------+-----+
| id|      name|           email|age|  salary| hire_date|is_manager|score|
+---+----------+----------------+---+--------+----------+----------+-----+
|  1|Employee_1|emp1@company.com| 29|90842.89|2022-03-04|     false| 7.63|
|  2|Employee_2|emp2@company.com| 56|77510.37|2022-08-03|     false|  4.8|
|  3|Employee_3|emp3@company.com| 27|79645.95|2015-05-03|     false| 2.79|
|  4|Employee_4|emp4@company.com| 56|98271.17|2022-04-15|     false| 5.04|
|  5|Employee_5|emp5@company.com| 22|75652.47|2018-02-13|     false|  3.5|
+---+----------+----------------+---+--------+----------+-------

In [7]:
# What happens with wrong schema (wider type = ok, narrower = nulls)
wrong_schema = StructType([
    StructField("id",       IntegerType()),
    StructField("name",     StringType()),
    StructField("email",    StringType()),
    StructField("age",      StringType()),    # STRING instead of INT → ok, reads as string
    StructField("salary",   StringType()),    # STRING instead of DOUBLE → ok
    StructField("hire_date",StringType()),    # STRING instead of DATE → ok
    StructField("is_manager",StringType()),
    StructField("score",    IntegerType()),   # INT instead of FLOAT → nulls for decimals!
])

df_wrong = spark.read.csv(csv_path, header=True, schema=wrong_schema)
print("Wrong schema (score as INT instead of FLOAT) — decimal scores become null:")
df_wrong.select("id","score").show(5)
print(f"Null scores: {df_wrong.filter(F.col('score').isNull()).count()} (all decimal scores lost!)")

Wrong schema (score as INT instead of FLOAT) — decimal scores become null:
+---+-----+
| id|score|
+---+-----+
|  1| NULL|
|  2| NULL|
|  3| NULL|
|  4| NULL|
|  5| NULL|
+---+-----+
only showing top 5 rows
Null scores: 100000 (all decimal scores lost!)


## Step 5 — Schema Evolution: Multiple CSV Versions

When CSV schema changes over time (new columns added),
you need to handle reading both old and new files.


In [8]:
# Version 1: original schema
csv_v1 = """order_id,customer,amount
1,Alice,100.0
2,Bob,200.0"""

# Version 2: added 'discount' and 'region' columns
csv_v2 = """order_id,customer,amount,discount,region
3,Carol,300.0,0.1,EMEA
4,Dave,400.0,0.0,AMER"""

pathlib.Path(f"{DATA_DIR}/orders_v1.csv").write_text(csv_v1)
pathlib.Path(f"{DATA_DIR}/orders_v2.csv").write_text(csv_v2)

# Strategy 1: read each version with its own schema
schema_v1 = StructType([
    StructField("order_id", IntegerType()),
    StructField("customer", StringType()),
    StructField("amount",   DoubleType()),
])

schema_v2 = StructType([
    StructField("order_id", IntegerType()),
    StructField("customer", StringType()),
    StructField("amount",   DoubleType()),
    StructField("discount", DoubleType()),
    StructField("region",   StringType()),
])

df_v1 = spark.read.csv(f"{DATA_DIR}/orders_v1.csv", header=True, schema=schema_v1) \
             .withColumn("discount", F.lit(None).cast("double")) \
             .withColumn("region",   F.lit("UNKNOWN"))
df_v2 = spark.read.csv(f"{DATA_DIR}/orders_v2.csv", header=True, schema=schema_v2)

combined = df_v1.union(df_v2)
print("Combined v1 + v2 (schema union):")
combined.show()

# Strategy 2: use wider schema for both (nulls for missing columns)
wide_schema = schema_v2  # v2 has all columns
df_v1_wide = spark.read.csv(f"{DATA_DIR}/orders_v1.csv",
                             header=True, schema=wide_schema)
print("\nv1 with wide schema (missing columns = null):")
df_v1_wide.show()

Combined v1 + v2 (schema union):
+--------+--------+------+--------+-------+
|order_id|customer|amount|discount| region|
+--------+--------+------+--------+-------+
|       1|   Alice| 100.0|    NULL|UNKNOWN|
|       2|     Bob| 200.0|    NULL|UNKNOWN|
|       3|   Carol| 300.0|     0.1|   EMEA|
|       4|    Dave| 400.0|     0.0|   AMER|
+--------+--------+------+--------+-------+


v1 with wide schema (missing columns = null):
+--------+--------+------+--------+------+
|order_id|customer|amount|discount|region|
+--------+--------+------+--------+------+
|       1|   Alice| 100.0|    NULL|  NULL|
|       2|     Bob| 200.0|    NULL|  NULL|
+--------+--------+------+--------+------+



## Summary

### When to use inferSchema
- Quick exploration / notebooks
- One-off analysis where speed doesn't matter
- Small files (< 100 MB)

### Always use explicit schema when
- Production pipelines
- Files > 100 MB
- Automated jobs
- When types must be guaranteed

### Schema building quick reference
```python
# StructType
StructType([StructField("col", IntegerType(), nullable=True)])

# DDL string
StructType.fromDDL("col INT, name STRING NOT NULL")

# From JSON file
StructType.fromJson(json.loads(Path("schema.json").read_text()))

# From existing DataFrame
existing_df.schema
```
